In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

# If the notebook is running from the analysis folder,
# move up one directory to the project root.
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

athletes = pd.read_csv(DATA_DIR / "athlete_events.csv")
noc_regions = pd.read_csv(DATA_DIR / "noc_regions.csv")

print("Athlete dataset shape:", athletes.shape)
print("NOC mapping shape:", noc_regions.shape)

Athlete dataset shape: (271116, 15)
NOC mapping shape: (230, 3)


In [3]:
display(athletes.head())
display(athletes.sample(5, random_state=42))

print("Column names:")
print(athletes.columns.tolist())

print("\nData types and non-null counts:")
athletes.info()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
195789,98308,Khalid Raghib,M,22.0,180.0,70.0,Morocco,MAR,1992 Summer,1992,Summer,Barcelona,Football,Football Men's Football,NaN
243016,121701,Mrioara Trac (-Curelea),F,21.0,172.0,68.0,Romania,ROU,1984 Summer,1984,Summer,Los Angeles,Rowing,Rowing Women's Coxed Eights,Silver
265866,133023,Martin Laumann Ylven,M,21.0,190.0,92.0,Norway,NOR,2010 Winter,2010,Winter,Vancouver,Ice Hockey,Ice Hockey Men's Ice Hockey,NaN
144505,72524,"Wilhelm Bernhard Adolf Emil ""Willy"" Ltzow",M,19.0,NaN,NaN,Germany,GER,1912 Summer,1912,Summer,Stockholm,Swimming,Swimming Men's 400 metres Breaststroke,NaN
150954,75751,Jess Martnez Tejeda,M,20.0,163.0,48.0,Mexico,MEX,1996 Summer,1996,Summer,Atlanta,Boxing,Boxing Men's Light-Flyweight,NaN


Column names:
['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal']

Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 56.6 MB


In [4]:
quality_report = pd.DataFrame({
    "data_type": athletes.dtypes.astype(str),
    "missing_count": athletes.isna().sum(),
    "missing_percent": (
        athletes.isna().mean() * 100
    ).round(2),
    "unique_values": athletes.nunique()
})

display(
    quality_report.sort_values(
        "missing_percent", ascending=False
    )
)

print("Exact duplicate rows:", athletes.duplicated().sum())
print("Unique athlete IDs:", athletes["ID"].nunique())
print("Year range:", athletes["Year"].min(), "to", athletes["Year"].max())
print("Medal values:", athletes["Medal"].value_counts(dropna=False))

,data_type,missing_count,missing_percent,unique_values
Medal,str,231333,85.33,3
Weight,float64,62875,23.19,220
Height,float64,60171,22.19,95
Age,float64,9474,3.49,74
ID,int64,0,0.00,135571
Name,str,0,0.00,134732
Sex,str,0,0.00,2
Team,str,0,0.00,1184
NOC,str,0,0.00,230
Games,str,0,0.00,51


Exact duplicate rows: 1385
Unique athlete IDs: 135571
Year range: 1896 to 2016
Medal values: Medal
NaN       231333
Gold       13372
Bronze     13295
Silver     13116
Name: count, dtype: int64


In [5]:
display(noc_regions.head())
print(noc_regions.info())

print("Duplicate NOC codes:", noc_regions["NOC"].duplicated().sum())

unmatched_nocs = (
    set(athletes["NOC"].dropna().unique())
    - set(noc_regions["NOC"].dropna().unique())
)

print("NOC codes without a mapping:", sorted(unmatched_nocs))

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


<class 'pandas.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   NOC     230 non-null    str  
 1   region  227 non-null    str  
 2   notes   21 non-null     str  
dtypes: str(3)
memory usage: 8.4 KB
None
Duplicate NOC codes: 0
NOC codes without a mapping: ['SGP']


In [6]:
duplicate_rows = athletes[
    athletes.duplicated(keep=False)
].sort_values(
    ["ID", "Year", "Event"]
)

display(duplicate_rows.head(20))

print(
    "Rows involved in exact duplicates:",
    len(duplicate_rows)
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
1251,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN
1252,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN
4281,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN
4282,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN
4283,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN
4863,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN
4864,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN
4861,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN
4862,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN
4866,2777,Hermann Reinhard Alker,M,51.0,NaN,NaN,Germany,GER,1936 Summer,1936,Summer,Berlin,Art Competitions,"Art Competitions Mixed Architecture, Unknown E...",NaN


Rows involved in exact duplicates: 1997


In [7]:
event_key = ["ID", "Games", "Event"]

repeated_events = athletes[
    athletes.duplicated(
        subset=event_key,
        keep=False
    )
].sort_values(event_key)

display(repeated_events.head(20))

print(
    "Rows in repeated athlete-event groups:",
    len(repeated_events)
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
1251,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN
1252,704,Dsir Antoine Acket,M,27.0,NaN,NaN,Belgium,BEL,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN
4017,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Silver
4018,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
4281,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN
4282,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN
4283,2449,William Truman Aldrich,M,48.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Painting, Drawings And ...",NaN
4863,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN
4864,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Architect...",NaN
4861,2777,Hermann Reinhard Alker,M,43.0,NaN,NaN,Germany,GER,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Architecture, Designs F...",NaN


Rows in repeated athlete-event groups: 2167


In [8]:
print(
    athletes.loc[
        athletes["NOC"] == "SGP",
        ["Team", "NOC", "Games", "Year"]
    ].drop_duplicates().sort_values("Year").head(20)
)

print(
    noc_regions[
        noc_regions["NOC"].isin(["SGP", "SIN"])
    ]
)

                Team  NOC        Games  Year
247372     Singapore  SGP  1948 Summer  1948
21961      Singapore  SGP  1952 Summer  1952
80986        Rika II  SGP  1956 Summer  1956
38075      Singapore  SGP  1956 Summer  1956
44376   June Climene  SGP  1960 Summer  1960
122792     Singapore  SGP  1960 Summer  1960
41428      Singapore  SGP  1968 Summer  1968
38312      Singapore  SGP  1972 Summer  1972
39082      Singapore  SGP  1976 Summer  1976
7659       Singapore  SGP  1984 Summer  1984
7663       Singapore  SGP  1988 Summer  1988
578        Singapore  SGP  1992 Summer  1992
579        Singapore  SGP  1996 Summer  1996
27339      Singapore  SGP  2000 Summer  2000
27341      Singapore  SGP  2004 Summer  2004
235895   Singapore-1  SGP  2004 Summer  2004
108582   Singapore-2  SGP  2004 Summer  2004
33525      Singapore  SGP  2008 Summer  2008
39574      Singapore  SGP  2012 Summer  2012
2719       Singapore  SGP  2016 Summer  2016
     NOC     region notes
178  SIN  Singapore   NaN


In [9]:
# Step 1: Remove exact duplicate rows
athletes_clean = athletes.drop_duplicates().copy()

print("Original rows:", len(athletes))
print("Cleaned rows:", len(athletes_clean))
print("Rows removed:", len(athletes) - len(athletes_clean))

# Step 2: Prepare a corrected copy of the NOC mapping
noc_regions_clean = noc_regions.copy()

if "SGP" not in noc_regions_clean["NOC"].values:
    singapore_mapping = pd.DataFrame([{
        "NOC": "SGP",
        "region": "Singapore",
        "notes": "Added during data cleaning"
    }])

    noc_regions_clean = pd.concat(
        [noc_regions_clean, singapore_mapping],
        ignore_index=True
    )

# Step 3: Merge country/region names into athlete records
athletes_clean = athletes_clean.merge(
    noc_regions_clean[["NOC", "region"]],
    on="NOC",
    how="left",
    validate="many_to_one"
)

athletes_clean = athletes_clean.rename(
    columns={"region": "Region"}
)

# Step 4: Validate the results
print("\nCleaned dataset shape:", athletes_clean.shape)
print("Remaining exact duplicates:", athletes_clean.duplicated().sum())
print("Missing region values:", athletes_clean["Region"].isna().sum())
print(
    "Unmapped NOC codes:",
    sorted(
        set(athletes_clean.loc[
            athletes_clean["Region"].isna(), "NOC"
        ].unique())
    )
)

# Step 5: Save the processed dataset
output_path = PROJECT_ROOT / "data" / "processed" / "athlete_events_clean.csv"

athletes_clean.to_csv(output_path, index=False)

print("\nSaved cleaned dataset to:", output_path)

Original rows: 271116
Cleaned rows: 269731
Rows removed: 1385

Cleaned dataset shape: (269731, 16)
Remaining exact duplicates: 0
Missing region values: 21
Unmapped NOC codes: ['ROT', 'TUV', 'UNK']

Saved cleaned dataset to: /Users/sasmita/Desktop/olympics-data-explorer/data/processed/athlete_events_clean.csv
